In [36]:
import os
import numpy as np
import ctypes
from tinygrad import Tensor, GlobalCounters, Context
from tinygrad.engine.realize import CompiledRunner
from tinygrad.runtime.ops_cpu import CPUProgram
from dataclasses import replace
# from keystone import Ks, KS_ARCH_ARM64, KS_MODE_LITTLE_ENDIAN

In [37]:
reduce_src = """
// data1 is 16M inputs
typedef float float4 __attribute__((aligned(32),vector_size(16)));
void reduce(float* restrict data0, float* restrict data1) {
  float4 acc0 = {0.0f, 0.0f, 0.0f, 0.0f};
  float4 acc1 = {0.0f, 0.0f, 0.0f, 0.0f};
  float4 acc2 = {0.0f, 0.0f, 0.0f, 0.0f};
  float4 acc3 = {0.0f, 0.0f, 0.0f, 0.0f};
  float4 acc4 = {0.0f, 0.0f, 0.0f, 0.0f};
  float4 acc5 = {0.0f, 0.0f, 0.0f, 0.0f};
  float4 acc6 = {0.0f, 0.0f, 0.0f, 0.0f};
  float4 acc7 = {0.0f, 0.0f, 0.0f, 0.0f};
  float* data1_1 = data1+4194304;
  float* data1_2 = data1+(4194304*2);
  float* data1_3 = data1+(4194304*3);
  for (int ridx0 = 0; ridx0 < 16777216/4; ridx0+=16) {
    float4 val0 = *(float4*)((data1+(ridx0+0)));
    float4 val1 = *(float4*)((data1+(ridx0+4)));
    float4 val2 = *(float4*)((data1+(ridx0+8)));
    float4 val3 = *(float4*)((data1+(ridx0+12)));
    acc0 += val0;
    acc1 += val1;
    acc2 += val2;
    acc3 += val3;
    val0 = *(float4*)((data1_1+(ridx0+0)));
    val1 = *(float4*)((data1_1+(ridx0+4)));
    val2 = *(float4*)((data1_1+(ridx0+8)));
    val3 = *(float4*)((data1_1+(ridx0+12)));
    acc4 += val0;
    acc5 += val1;
    acc6 += val2;
    acc7 += val3;
    val0 = *(float4*)((data1_2+(ridx0+0)));
    val1 = *(float4*)((data1_2+(ridx0+4)));
    val2 = *(float4*)((data1_2+(ridx0+8)));
    val3 = *(float4*)((data1_2+(ridx0+12)));
    acc0 += val0;
    acc1 += val1;
    acc2 += val2;
    acc3 += val3;
    val0 = *(float4*)((data1_3+(ridx0+0)));
    val1 = *(float4*)((data1_3+(ridx0+4)));
    val2 = *(float4*)((data1_3+(ridx0+8)));
    val3 = *(float4*)((data1_3+(ridx0+12)));
    acc4 += val0;
    acc5 += val1;
    acc6 += val2;
    acc7 += val3;
  }
  float4 out = acc0+acc1+acc2+acc3+acc4+acc5+acc6+acc7;
  *(data0+0) = out[0]+out[1]+out[2]+out[3];
}
"""

In [38]:
with Context(DEBUG=7):
  a = Tensor(np_array:=(np.random.default_rng().random((4096, 4096), dtype=np.float32)-0.5), device="CPU").realize()
  with Context(SPLIT_REDUCEOP=0):
    # TODO: make it easy to alter the OptOps for a ScheduleItem
    GlobalCounters.reset()
    out = a.sum()
    sis = out.schedule()
    for i,ei in enumerate(sis):
      ei.lower()
      if i == 0:
        # change the source code
        prg_spec = ei.prg.p
        prg_spec = replace(prg_spec, name="reduce", src=reduce_src, lib=None)
        prg = CompiledRunner(prg_spec)
        # change the assembly
        #prg._prg = CPUProgram(prg_spec.name, arm_bytecode)
        print("buffer at:",hex(ei.bufs[1]._buf.va_addr))
        ei = replace(ei, prg=prg)
      ei.run()
    print(out.item())
    np.testing.assert_allclose(out.item(), np_array.sum(), atol=1, rtol=1e-4)


buffer: allocate 67108864 bytes on NPY
scheduled    1 kernels in     2.10 ms | CACHE MISS 302f9437 | 157 uops in cache
opened device CPU from pid:15937
buffer: allocate 67108864 bytes on CPU
*** CPU        1 copy   67.11M,     CPU <- NPY                  arg  2 mem   0.20 GB tm     13.04ms/    13.04ms (      0 GFLOPS    5|5      GB/s) 
buffer: deallocate 67108864 bytes on NPY
scheduled    1 kernels in     1.33 ms | CACHE MISS b0ff6c51 | 165 uops in cache
buffer: deallocate 67108864 bytes on METAL
c0 = UOp(Ops.DEFINE_GLOBAL, dtypes.float.ptr(1), (), 0)
c3 = UOp(Ops.DEFINE_GLOBAL, dtypes.float.ptr(16777216), (), 1)
c5 = UOp.range(4096, 0, AxisType.REDUCE)
c7 = UOp.range(4096, 1, AxisType.REDUCE)
c9 = c3.index((c5*UOp.const(dtypes.index, 4096)+c7))
c10 = c9.reduce(c5, c7, arg=Ops.ADD)
c11 = c0.index(UOp.const(dtypes.index, 0), ptr=True).store(c10)
ast = c11.sink(arg=KernelInfo(name='test', axis_types=(), dont_use_locals=False, applied_opts=(), opts_to_apply=None, estimates=None))
   0 Ops

In [39]:
print(4096*4096/8)

2097152.0


In [3]:
from tinygrad.uop.ops import UOp, Ops, KernelInfo, ShapeTracker, View
from tinygrad.codegen.opt import Opt, OptOps
from tinygrad.codegen.opt.postrange import Scheduler

ast = UOp(Ops.SINK, dtypes.void, arg=KernelInfo(local_dims=0, upcasted=0, dont_use_locals=False), src=(
  UOp(Ops.STORE, dtypes.void, arg=None, src=(
    UOp(Ops.DEFINE_GLOBAL, dtypes.float.ptr(), arg=0, src=()),
    UOp(Ops.VIEW, dtypes.void, arg=ShapeTracker(views=(View(shape=(4, 1), strides=(1, 0), offset=0, mask=None, contiguous=True),)), src=()),
    UOp(Ops.REDUCE_AXIS, dtypes.float, arg=(Ops.ADD, (1,)), src=(
      UOp(Ops.LOAD, dtypes.float, arg=None, src=(
        UOp(Ops.DEFINE_GLOBAL, dtypes.float.ptr(), arg=1, src=()),
        UOp(Ops.VIEW, dtypes.void, arg=ShapeTracker(views=(View(shape=(4, 4), strides=(4, 1), offset=0, mask=None, contiguous=True),)), src=()),)),)),)),))

unroll_opt = Opt(OptOps.UNROLL, 0, 4)

kernel = Scheduler(ast)
kernel.apply_opt(unroll_opt)
p = kernel.to_program()
print(p.src)

ImportError: cannot import name 'ShapeTracker' from 'tinygrad.uop.ops' (/Users/cobeliu/Developing/tinygrad/tinygrad/uop/ops.py)